# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

# Locate the project data file
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# Inspect distributions of key numeric fields
key_fields = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "days_since_update",
    "avg_position",
    "word_count",
]

available_fields = [col for col in key_fields if col in df.columns]

display(
    df[available_fields]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])
    .T
    .round(2)
)

print("\nMissing values (%):")
display((df[available_fields].isna().mean() * 100).round(2))

Dataset shape: (30000, 44)


,count,mean,std,min,25%,50%,75%,95%,99%,max
impressions_90d,30000.0,5200.37,16838.02,1.0,81.0,731.00,3615.25,22996.50,73505.83,517715.0
clicks_90d,30000.0,16.10,75.08,0.0,0.0,1.00,7.00,69.05,253.01,4178.0
ctr,30000.0,0.51,3.28,0.0,0.0,0.07,0.29,1.09,8.33,100.0
engagement_rate,30000.0,2.53,8.31,0.0,0.0,0.00,1.35,12.50,33.33,100.0
scroll_rate,29875.0,18.21,29.47,0.0,0.0,5.00,23.53,100.00,100.00,300.0
avg_position,30000.0,16.34,15.22,0.0,6.2,10.80,22.30,48.20,69.90,245.0
word_count,22301.0,3107.76,1452.38,8.0,2413.0,2877.00,3666.00,6173.00,7292.00,9546.0



Missing values (%):


impressions_90d     0.00
clicks_90d          0.00
ctr                 0.00
engagement_rate     0.00
scroll_rate         0.42
avg_position        0.00
word_count         25.66
dtype: float64

In [7]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def test_signal(signal, target="impressions_90d", bins=4):
    data = df[[signal, target]].dropna().copy()
    data["group"] = pd.qcut(
        data[signal], q=bins, duplicates="drop"
    )

    result = data.groupby("group", observed=True)[target].agg(
        ["count", "median", "mean"]
    ).round(2)

    print(f"\nSignal: {signal}")
    display(result)


# Test 1: Does higher search demand align with higher impressions?
test_signal("search_volume")

# Test 2: Does content age align with higher or lower impressions?
test_signal("content_age_days")

# Test 3: Does longer content align with higher impressions?
test_signal("word_count")


Signal: search_volume


,count,median,mean
group,,,
"(-0.001, 10.0]",18392,929.0,5547.99
"(10.0, 20.0]",2290,1006.5,5690.32
"(20.0, 74000.0]",6850,842.5,5796.31



Signal: content_age_days


,count,median,mean
group,,,
"(89.999, 132.0]",7518,768.0,5063.33
"(132.0, 236.0]",8128,796.5,5649.89
"(236.0, 333.0]",6917,601.0,5246.44
"(333.0, 564.0]",7437,733.0,4804.76



Signal: word_count


,count,median,mean
group,,,
"(7.999, 2413.0]",5576,91.0,1370.85
"(2413.0, 2877.0]",5586,1096.5,5396.79
"(2877.0, 3666.0]",5566,889.0,5983.65
"(3666.0, 9546.0]",5573,1495.0,7565.62


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test whether better search positions align with higher CTR
position_data = df[
    (df["avg_position"] > 0) &
    (df["ctr"].notna())
].copy()

position_result = (
    position_data
    .groupby("position_tier", observed=True)
    .agg(
        count=("ctr", "size"),
        median_position=("avg_position", "median"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .round(2)
)

display(position_result)

,count,median_position,median_ctr,mean_ctr
position_tier,,,,
deep,1319,61.0,0.00,0.15
page_1,11814,6.6,0.16,0.65
page_3_5,7242,28.9,0.03,0.22
striking,7304,13.9,0.11,0.32
top_3,1116,2.2,0.00,2.76


**Flag-linked test — Position tier and CTR: MIXED**

The data shows some support for the assumption that better search positions are associated with higher CTR. However, the pattern is not consistent across all position tiers. The top_3 group has a median CTR of 0.00 despite a much higher mean of 2.76, suggesting a skewed distribution. Position tier alone is therefore not a reliable indicator of CTR performance.

Content teams should treat search volume, content age, word count, and position tier as supporting signals rather than guaranteed indicators of performance. These observed relationships are useful for prioritizing human review, but decisions should consider multiple metrics and the context of each page.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final audit checks
print("Signal audit completed.")
print("Dataset rows:", len(df))
print("Signal tests:", 3)
print("Flag-linked test verdict: MIXED")
print("Trend-derived fields were not used as test features.")

Signal audit completed.
Dataset rows: 30000
Signal tests: 3
Flag-linked test verdict: MIXED
Trend-derived fields were not used as test features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.